<a href="https://colab.research.google.com/github/igMoreira/claude-cert-notebooks/blob/main/build_with_claude_api/class_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install Anthropic
!pip install python-dotenv

In [5]:
from anthropic import Anthropic
from dotenv import load_dotenv
import os

load_dotenv()

MODEL = 'claude-haiku-4-5-20251001'
#MODEL = 'claude-sonnet-5'
MAX_TOKENS = 1000
API_KEY = os.getenv('CLAUDE_API_KEY')
client = Anthropic(api_key=API_KEY)


In [6]:
import json

def generate_dataset():
    return """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "type": "Python" | "JSON" | "Regex",
    "solution_criteria": "A clear description of the most important points for the required solution, memory, cpu, timeout, maintainability, etc"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

def user_message(messages, text):
  messages.append({'role': 'user', 'content': text})
  return messages

def assistant_message(messages, text):
  messages.append({'role': 'assistant', 'content': text})
  return messages

def chat(messages, system=None, stop_sequences=None):
  params = {
      'model':MODEL,
      'max_tokens':MAX_TOKENS,
      'messages':messages
  }
  if system:
    params['system'] = system
  if stop_sequences:
    params['stop_sequences'] = stop_sequences

  response = client.messages.create(**params)
  answer = response.content[0].text
  assistant_message(messages, answer)
  return answer

In [7]:
messages = []
user_message(messages, generate_dataset())
assistant_message(messages, "```json")
output = chat(messages, stop_sequences=['```'])
dataset = json.loads(output)
with open('dataset.json', 'w') as f:
  json.dump(dataset, f, indent=2)

In [8]:
import json, re, ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_by_code(testcase, output):
  if testcase['type'] == 'Python':
    score = validate_python(output)
  elif testcase['type'] == 'JSON':
    score = validate_json(output)
  elif testcase['type'] == 'Regex':
    score = validate_regex(output)
  return score

def grade_by_model(testcase, output):
  # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.

    Task: {testcase['task']}
    Solution criteria: {testcase['solution_criteria']}
    Solution to evaluate:
    <solution>
    {output}
    </solution>

    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    messages = []
    user_message(messages, eval_prompt)
    assistant_message(messages, "```json")

    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

def run_prompt(testcase):
  prompt = f"""
  Solve the following task:

  {testcase["task"]}

  * Respond only with Python, JSON, or a plain Regex
  * Do not add any comments or commentary or explanation
  """
  messages = []
  user_message(messages, prompt)
  assistant_message(messages, "```code")
  answer = chat(messages, stop_sequences=['```'])
  return answer

def run_test_case(testcase):
  output = run_prompt(testcase)
  #TODO: grading
  model_grade = grade_by_model(testcase, output)
  model_score = model_grade['score']
  reasoning = model_grade['reasoning']
  code_score = grade_by_code(testcase, output)
  score = (model_score + code_score) / 2

  return {
      'score': score,
      'output': output,
      'testcase': testcase,
      'reasoning': reasoning
  }

def run_eval(dataset):
  results = []
  for data in dataset:
    results.append(run_test_case(data))
  return json.dumps(results, indent=2)

In [9]:
from statistics import mean

with open('dataset.json', 'r') as f:
  dataset = json.load(f)
results = json.loads(run_eval(dataset))
avg_score = mean([result['score'] for result in results])
print(f"Average score: {avg_score}")
print(results)

Average score: 7.333333333333333
[{'score': 6.5, 'output': '\nimport json\nimport re\nimport sys\n\ndef extract_s3_buckets(template):\n    buckets = set()\n    \n    if isinstance(template, str):\n        try:\n            template = json.loads(template)\n        except json.JSONDecodeError:\n            pass\n    \n    if isinstance(template, dict):\n        for key, value in template.items():\n            if key == "BucketName" and isinstance(value, str):\n                buckets.add(value)\n            elif key == "Bucket" and isinstance(value, str):\n                buckets.add(value)\n            elif isinstance(value, (dict, list)):\n                buckets.update(extract_s3_buckets(value))\n    elif isinstance(template, list):\n        for item in template:\n            buckets.update(extract_s3_buckets(item))\n    \n    return buckets\n\nif __name__ == "__main__":\n    input_data = sys.stdin.read()\n    result = extract_s3_buckets(input_data)\n    print(json.dumps(sorted(list(r